# Musicm8 — one-click Colab training

This notebook is designed to survive Colab runtime resets. **After a reset, run the big cell below once.** It mounts Drive, refreshes the GitHub repo, installs dependencies, rebuilds the runtime variables, reuses cached tokens, resumes `latest.pt` if present, and generates a sample when training finishes.

> Colab still requires a GPU runtime. The notebook requests a GPU in metadata, but Colab may still ask you to select **Runtime → Change runtime type → GPU**.


In [ ]:
# ============================================================
# MUSICM8 — ONE CLICK SETUP / TOKENIZE / TRAIN / RESUME / SAMPLE
# Run this cell after every Colab runtime reset.
# ============================================================

import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

def run_cmd(cmd, *, cwd=None):
    cmd = [str(x) for x in cmd]
    print("\n$", " ".join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {' '.join(cmd)}"
        )
    return result

# ---------- Drive ----------
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ---------- Fresh runtime copy of the GitHub code ----------
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"
REPO_DIR = Path("/content/Musicm8")

if (REPO_DIR / ".git").exists():
    print("Refreshing Musicm8 from GitHub...")
    run_cmd(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"])
    run_cmd(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"])
else:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
print("Repo:", REPO_DIR)

# ---------- Dependencies ----------
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

# Import torch only after dependencies are ready.
import torch

print("\nPython:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is connected. In Colab choose Runtime > Change runtime type > GPU, "
        "then run THIS SAME CELL again."
    )

print("GPU:", torch.cuda.get_device_name(0))

# ---------- Persistent settings in Google Drive ----------
DRIVE_ROOT = Path("/content/drive/MyDrive/Musicm8")
AUDIO_DIR = DRIVE_ROOT / "audio"
WORK_DIR = DRIVE_ROOT / "work"
MANIFEST = DRIVE_ROOT / "manifest.jsonl"

# Keep this in sync with tokenize_dataset.py choices.
CODEC = "encodec24"

CLIP_SECONDS = 8
STRIDE_SECONDS = 8
STEPS = 2000
BATCH_SIZE = 2
GRAD_ACCUM = 2
SAVE_EVERY = 250
RUN_NAME = "tiny-overfit"

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

TOKENS_DIR = WORK_DIR / f"tokens-{CODEC}"
INDEX = TOKENS_DIR / "index.jsonl"
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
LATEST = RUN_DIR / "latest.pt"
SAMPLE = WORK_DIR / "sample.wav"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("\n✅ WORK_DIR :", WORK_DIR)
print("✅ AUDIO_DIR:", AUDIO_DIR)
print("✅ CODEC    :", CODEC)
print("✅ RUN_NAME :", RUN_NAME)
print("✅ RUN_DIR  :", RUN_DIR)

# ---------- Audio + manifest ----------
exts = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac", ".opus"}
audio_files = sorted(
    p for p in AUDIO_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in exts
)
print(f"✅ Audio files found: {len(audio_files)}")

if not audio_files:
    raise FileNotFoundError(f"No audio files found in {AUDIO_DIR}")

if not MANIFEST.exists() or MANIFEST.stat().st_size == 0:
    with MANIFEST.open("w", encoding="utf-8") as f:
        for p in audio_files:
            caption = p.stem.replace("_", " " ).replace("-", " " )
            row = {"audio": str(p), "caption": f"music track, {caption}"}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("✅ Created manifest:", MANIFEST)
else:
    print("✅ Using existing manifest:", MANIFEST)

# ---------- Tokenize only when cache is missing ----------
if INDEX.exists() and INDEX.stat().st_size > 0:
    print("✅ Token cache already exists:", INDEX)
else:
    # Remove a partial failed cache so a retry starts cleanly.
    shutil.rmtree(TOKENS_DIR, ignore_errors=True)

    tokenize_cmd = [
        sys.executable,
        "tokenize_dataset.py",
        "--manifest", MANIFEST,
        "--out", TOKENS_DIR,
        "--codec", CODEC,
        "--channels", "1",
        "--clip-seconds", str(CLIP_SECONDS),
        "--stride-seconds", str(STRIDE_SECONDS),
        "--keep-tail",
        "--device", "cuda",
    ]
    print("\n🎵 Tokenizing audio...")
    run_cmd(tokenize_cmd)

if not INDEX.exists() or INDEX.stat().st_size == 0:
    raise RuntimeError(f"Tokenization finished but no usable index was created at {INDEX}")

print("✅ Tokenization ready:", INDEX)

# ---------- Train / resume ----------
train_cmd = [
    sys.executable,
    "train.py",
    "--data", INDEX,
    "--config", "configs/v2-tiny.json",
    "--out", RUN_DIR,
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--steps", str(STEPS),
    "--save-every", str(SAVE_EVERY),
    "--num-workers", "2",
    "--device", "cuda",
]

if LATEST.exists():
    train_cmd += ["--resume", LATEST]
    print("\n♻️ Resuming checkpoint:", LATEST)
else:
    print("\n🚀 Starting a fresh training run")

run_cmd(train_cmd)

if not LATEST.exists():
    raise RuntimeError(f"Training ended but checkpoint was not found at {LATEST}")

print("✅ Checkpoint:", LATEST)

# ---------- Generate a sample ----------
PROMPT = "dark atmospheric electronic music with deep bass and wide synth pads"
generate_cmd = [
    sys.executable,
    "generate.py",
    "--checkpoint", LATEST,
    "--prompt", PROMPT,
    "--seconds", "8",
    "--seed", "42",
    "--out", SAMPLE,
]

print("\n🎧 Generating sample...")
run_cmd(generate_cmd)

from IPython.display import Audio, display
display(Audio(str(SAMPLE)))

print("\n✅ MUSICM8 COMPLETE")
print("Checkpoint:", LATEST)
print("Sample    :", SAMPLE)


## Optional: quick status check
Run this if you only want to see what has been saved in Drive without starting training.


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/Musicm8")
work = root / "work"
print("Audio folder :", root / "audio")
print("Manifest     :", root / "manifest.jsonl")
print("Token indexes:", list(work.glob("tokens-*/index.jsonl")) if work.exists() else [])
print("Checkpoints  :", list(work.glob("runs/*/latest.pt")) if work.exists() else [])
print("Samples      :", list(work.glob("*.wav")) if work.exists() else [])
